# Reading Stop-signal task data

1. Read data from export folder
2. Create event dataframe to store event type, latencies, SRI, SSD, stop type, response type 
3. Add info on baseline value, apply baseline correction and add info on artifact rejection -- channel-wise. This returns dictionary where key is a channel name, and value is an event dataframe
4. Apply drop log to non-baselined data and compare current export against BVA-based artifact rejection
5. Check for the response before stop signal marker sequences
5. Calculate centered, standardized, and normalized values of SSD and SRI

In [3]:
import logging
from mne.utils import set_log_file
from preprocessing import *
import warnings
import os
import pickle
# Suppress FutureWarning
warnings.simplefilter(action='ignore', category=FutureWarning)

Define output dir and loggers

In [2]:
output_dir = '../data/output'

In [3]:
######## PREPROCESSING ##############################################
# Create a custom logger for preprocessing INFO
logger_preprocessing_info = logging.getLogger('preprocessing_info')
logger_preprocessing_info.setLevel(logging.INFO)

######## ERRORS ##############################################
# Create a custom logger for errors
logger_errors_info = logging.getLogger('errors')
logger_errors_info.setLevel(logging.INFO)

######## PREPROCESSING ##############################################
# Create a file handler for preprocessing and set the level to INFO
file_handler_preprocessing = logging.FileHandler(f'{output_dir}/preprocessing.txt')
file_handler_preprocessing.setLevel(logging.INFO)

# Create a formatter and add it to the file handler for preprocessing
formatter_preprocessing = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
file_handler_preprocessing.setFormatter(formatter_preprocessing)

# Add the file handler for method A to the logger for preprocessing
logger_preprocessing_info.addHandler(file_handler_preprocessing)

######## ERRORS ##############################################
# Create a file handler for errors and set the level to INFO
file_handler_errors = logging.FileHandler(f'{output_dir}/errors.txt')
file_handler_errors.setLevel(logging.INFO)

# Create a formatter and add it to the file handler for errors
formatter_errors = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
file_handler_errors.setFormatter(formatter_errors)

# Add the file handler for method A to the logger for preprocessing
logger_errors_info.addHandler(file_handler_errors)

##### MNE ###################################################
# Create logger for MNE logs
logger_f_name = f'{output_dir}/MNE-logs.txt'
set_log_file(fname=logger_f_name, output_format="%(asctime)s - %(message)s", overwrite=None)

Define picks for the export

In [4]:
picks = [
    'Fp1',
    'AF3',
    'F7',
    'F3',
    'FC1',
    'FC5',
    'T7',
    'C3',
    'CP1',
    'CP5',
    'P7',
    'P3',
    'Pz',
    'PO3',
    'O1',
    'Oz',
    'O2',
    'PO4',
    'P4',
    'P8',
    'CP6',
    'CP2',
    'C4',
    'T8',
    'FC6',
    'FC2',
    'F4',
    'F8',
    'AF4',
    'Fp2',
    'Fz',
    'Cz',
]

## Read and preprocess data

In [5]:
def extract_ids_from_filenames(filenames):
    """
    Extract IDs from a list of filenames. ID is everything before the last underscore '_'.
    
    Parameters:
    filenames (list): A list of filenames.
    
    Returns:
    list: A list of extracted IDs.
    """
    ids = []
    for filename in filenames:
        # Remove the file extension
        name_without_ext = os.path.splitext(filename)[0]
        # Find the last underscore
        last_underscore_index = name_without_ext.rfind('_')
        if last_underscore_index != -1:
            # Extract ID before the last underscore
            id_part = name_without_ext[:last_underscore_index]
            ids.append(id_part)
        else:
            # If no underscore found, consider the whole name as ID
            ids.append(name_without_ext)
    return ids

In [6]:
data_dir = '../data/Export SST N=54_Correct filtr 0.1-15 Hz'
vhdr_files = sorted([f for f in os.listdir(data_dir) if f.endswith('.vhdr')])
id_list = extract_ids_from_filenames(vhdr_files)

In [7]:
transforms = [
    ('center', lambda x: x - x.mean()),
    ('standardized', lambda x: x / x.std()),
    ('normalized', lambda x: (x - x.mean()) / (x - x.mean()).std())
]

In [ ]:
data_dir = '../data/Export SST N=54_Correct filtr 0.1-15 Hz'
# old_data_dir = 'Export SST N=54_Correct filtr 0.1'

for id in id_list:
    current_suffix = 'EOG4'
    # old_suffix = 'Artif Rej 75'
    
    tmin = -0.2
    tmax = 1.58
    
    try:
    
        _, drop_log, epochs = preprocess(
            file_name = f'{id}_{current_suffix}',
            dir_name = data_dir,
            picks=picks,
            tmin=tmin,
            tmax=tmax,
            fill_missing = False,
            columns_to_transform=['ssd', 'sri'],
            transforms=transforms,
            output_dir=output_dir,
            logger_preprocessing_info=logger_preprocessing_info,
            logger_errors_info=logger_errors_info,
            save_output=True
        )
        
        bad_trails = np.unique([trial for trials in drop_log.values() for trial in trials])
        epochs_clean = epochs.copy().drop(bad_trails)
        
        # epochs_old =  read_old_data(
        #     file_name = f'{id}_{old_suffix}',
        #     dir_name = old_data_dir,
        #     tmin = tmin,
        #     tmax=tmax,
        # )
        # 
        # # plot old export epochs against new export epochs
        # channels_of_interest = ['Cz', 'Fz']
        # plot_old_against_new(
        #     epochs_clean,
        #     epochs_old,
        #     channels_of_interest,
        #     tmin=tmin,
        #     tmax=tmax,
        #     id = id
        # )
    except Exception as e:
        logger_errors_info.info(f"{e}")
        
    logger_preprocessing_info.info(f'\n')
    logger_errors_info.info(f'\n') 

In [4]:
with open('../data/output/AB0407_SST14_EOG4.pickle', 'rb') as handle:
    events_ = pickle.load(handle)

In [90]:
events_['Fp1']

,index,latency,duration,id,event,event_general,type,ssd,response_type,stop_type,sri,baseline,bad,ssd_center,ssd_standardized,ssd_normalized,sri_center,sri_standardized,sri_normalized
0,1,13,0,8,Stimulus/ 3-B-STOP2-SE-R,go/stop/2/SE,go,150.0,n-a,SE,NaN,167.146583,True,NaN,NaN,NaN,NaN,NaN,NaN
1,1,13,0,8,Stimulus/ 3-B-STOP2-SE-R,go/stop/2/SE,baseline,150.0,n-a,SE,NaN,167.146583,True,NaN,NaN,NaN,NaN,NaN,NaN
2,2,23,0,52,Stimulus/ 3-STOP2-SE-R,stop/2/SE,stop,150.0,n-a,SE,NaN,167.146583,True,NaN,NaN,NaN,NaN,NaN,NaN
3,3,30,0,36,Stimulus/ 3-R-B-STOP2-SE-R,response/incorrect/2,response_stop,150.0,incorrect,n-a,109.375,167.146583,True,NaN,NaN,NaN,NaN,NaN,NaN
5,4,128,0,3,Stimulus/ 3-B-STOP1-SE-L,go/stop/1/SE,go,100.0,n-a,SE,NaN,-28.836250,False,-15.671642,3.05946,-0.479468,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
626,626,22553,0,5,Stimulus/ 3-B-STOP1-SS-L,go/stop/1/SS,baseline,100.0,n-a,SS,NaN,-7.361333,False,-15.671642,3.05946,-0.479468,NaN,NaN,NaN
628,628,22559,0,49,Stimulus/ 3-STOP1-SS-L,stop/1/SS,stop,100.0,n-a,SS,NaN,-7.361333,False,-15.671642,3.05946,-0.479468,NaN,NaN,NaN
629,629,22668,0,1,Stimulus/ 3-B-NOSTOP-L,go/nostop,go,NaN,n-a,n-a,NaN,-4.657833,False,NaN,NaN,NaN,NaN,NaN,NaN
630,629,22668,0,1,Stimulus/ 3-B-NOSTOP-L,go/nostop,baseline,NaN,n-a,n-a,NaN,-4.657833,False,NaN,NaN,NaN,NaN,NaN,NaN
